<a href="https://colab.research.google.com/github/Asmitpremkumar16/Customer_Churn/blob/main/Churn_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/Asmitpremkumar16/Customer_Churn.git

In [ ]:
%cd Customer_Churn/

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [ ]:
data= pd.read_csv(r"/content/Customer_Churn/churn modelling.csv")
data.head()

In [ ]:
data.isna().sum()

In [ ]:
data.duplicated().sum()

In [ ]:
data["Exited"].value_counts(normalize=True) * 100

In [ ]:
data.groupby('Geography')['Exited'].mean() * 100

In [ ]:
data.groupby('Gender')['Exited'].mean() * 100

In [ ]:
data.groupby(['Geography','Gender'])['Exited'].mean() * 100

In [ ]:
data.groupby(['Gender','IsActiveMember'])['Exited'].mean() * 100

In [ ]:
data.groupby(['NumOfProducts','IsActiveMember'])['Exited'].mean() * 100

In [ ]:
data.groupby(['HasCrCard'])['Exited'].mean() * 100

In [ ]:
fig1= px.histogram(data, x= "Geography", color= "IsActiveMember",facet_col= "Exited", barmode= "group", title= "Geography by Exited", text_auto= True)
fig1.update_traces(textposition= 'outside')
fig1.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data= data, x= "Geography", hue= "Exited")
plt.show()

In [ ]:
df= data[["CreditScore","Age","Tenure","Balance","NumOfProducts","HasCrCard","IsActiveMember","EstimatedSalary","Exited"]].corr()
sns.heatmap(df, annot= True, fmt= ".2f")
plt.show()

In [ ]:
fig2= px.imshow(df, text_auto= ".2f",title= "Correlation Heatmap")
fig2.update_layout(width= 700, height= 600)
fig2.show()

In [ ]:
data['Age'].describe()

In [ ]:
fig5= px.histogram(data, x= "Age", color= "Exited", barmode= 'overlay', title= "Age Distribution Churned vs Retained", opacity= 0.6)
fig5.show()

In [ ]:
fig6= px.histogram(data, x= "EstimatedSalary", color= "Exited", opacity= 0.6, title= "Churn across different Balance")
fig6.show()

In [ ]:
import torch
import torch.nn as nn
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.base import BaseEstimator, TransformerMixin

In [ ]:
data= data.drop(columns= ['RowNumber','CustomerId','Surname'], axis= 1)
data.head()

In [ ]:
X= data.drop(columns= ['Exited'], axis= 1)
y= data['Exited']

X_train, X_test, y_train, y_test= train_test_split(X, y, test_size= 0.2, random_state= 42, stratify= y)

In [ ]:
numeric_columns= ['CreditScore', 'Age','Tenure','Balance','EstimatedSalary']
categorical_columns= ['Geography','Gender']
pass_columns= ['NumOfProducts','HasCrCard','IsActiveMember']

preprocessor= ColumnTransformer(transformers=[
    ('num',StandardScaler(), numeric_columns),
    ('cat', OneHotEncoder(drop= 'first',sparse_output= False), categorical_columns),
    ('pass', 'passthrough',pass_columns)
])

In [ ]:
X_train_processed= preprocessor.fit_transform(X_train)
X_test_processed= preprocessor.transform(X_test)

In [ ]:
X_train_processed[1]

In [ ]:
sm= SMOTE(random_state= 42)

X_train_res, y_train_res= sm.fit_resample(X_train_processed, y_train)

In [ ]:
X_train_res[1], y_train_res.shape

In [ ]:
X_train_tensor= torch.tensor(X_train_res, dtype= torch.float32)
X_test_tensor= torch.tensor(X_test_processed, dtype= torch.float32)
y_train_tensor= torch.tensor(y_train_res.values, dtype= torch.float32)
y_test_tensor= torch.tensor(y_test.values, dtype= torch.float32)

In [ ]:
# MODEL BUILDING

class ChurnModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.network= nn.Sequential(
        nn.Linear(in_features= 11, out_features= 64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(in_features= 64, out_features= 128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(in_features= 128, out_features= 64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(in_features= 64, out_features= 32),
        nn.BatchNorm1d(32),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(in_features=32, out_features= 10),
        nn.ReLU(),
        nn.Linear(in_features=10, out_features= 1),
        nn.Sigmoid()
    )

  def forward(self, x):
    return self.network(x)

In [ ]:
model= ChurnModel()
loss_fn= nn.BCELoss()
optimizer= torch.optim.RMSprop(params= model.parameters(), lr= 0.01)

In [ ]:
# TRAIN PART

best_test_loss= float('inf')
patience= 50
counter= 0
best_weights= None

for epoch in range(200):
  model.train()

  optimizer.zero_grad()
  y_pred= model(X_train_tensor).squeeze()
  loss= loss_fn(y_pred, y_train_tensor)
  loss.backward()
  optimizer.step()

  model.eval()
  with torch.inference_mode():
    y_test_pred= model(X_test_tensor).squeeze()
    test_loss= loss_fn(y_test_pred, y_test_tensor)

  if test_loss < best_test_loss:
    best_test_loss= test_loss
    best_weights= model.state_dict().copy()
    counter = 0
  else:
    counter +=1

  if counter >= patience:
    print(f"Early stopping at Epoch {epoch + 1}")
    break

  if (epoch + 1) % 10 == 0:
    print(f"Epoch:{epoch + 1}|Train_Loss:{loss.item():.4f}|Test_Loss:{test_loss.item():.4f}")

In [ ]:
from sklearn.metrics import classification_report

model.eval()
with torch.inference_mode():
  y_pred_probs= model(X_test_tensor).squeeze()
  y_pred_labels= torch.round(y_pred_probs)

print(classification_report(y_test_tensor.numpy(), y_pred_labels.numpy()))

In [ ]:
model.load_state_dict(best_weights)

In [ ]:
torch.save(model.state_dict(), "ChurnModel.pth")

with open("preprocessor.pkl", "wb") as f:
  pickle.dump(preprocessor, f)

print("Saved successfully!")

In [ ]:
# from google.colab import files

# files.download("ChurnModel.pth")
# files.download("preprocessor.pkl")